# Before DuckDB: OLTP vs. OLAP

## Begin with the workload

Consider an online store.

The checkout service repeatedly performs small, time-sensitive operations:

```text
1. Create one order.
2. Reserve a few inventory items.
3. Record one payment.
4. Commit all changes together—or roll them all back.
5. Allow many customers to check out concurrently.
```

This is **online transaction processing (OLTP)**. A system such as MySQL with InnoDB is designed to serve many application connections performing short reads and writes against individual rows.

Now consider a business-analysis request:

```text
Scan five years of orders.
Join orders, products, and customers.
Group millions of rows by month and region.
Calculate revenue, growth, and average order value.
```

This is **online analytical processing (OLAP)**. DuckDB is designed to scan, join, filter, and aggregate large batches efficiently, including data stored in Parquet.

## High-level comparison

| Concern | MySQL with InnoDB: OLTP focus | DuckDB: OLAP focus |
|---|---|---|
| Typical unit of work | A few rows per transaction | Thousands to billions of rows per query |
| Typical operations | `INSERT`, point lookup, small `UPDATE`, small `DELETE` | Scan, join, aggregate, window function, bulk transformation |
| Deployment | Long-running client/server database | Embedded, in-process analytical database |
| Users and writers | Many independent application clients | Commonly one process, with multiple threads/connections inside it |
| Concurrency approach | Fine-grained InnoDB locking plus MVCC | MVCC and optimistic concurrency inside one writer process |
| Storage optimization | Row-oriented pages and indexes for record access | Vectorized, column-oriented analytical execution |
| Common data source | InnoDB application tables | Parquet, CSV, Arrow, pandas, and DuckDB tables |
| Main design goal | Correct, low-latency concurrent transactions | Fast analytical queries and data transformation |

## Important correction: DuckDB does support constraints

The presence of a primary key does **not** determine whether a database is OLTP or OLAP. DuckDB supports these table constraints:

```sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    email VARCHAR UNIQUE NOT NULL,
    status VARCHAR CHECK (status IN ('active', 'inactive'))
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    amount DECIMAL(12, 2) CHECK (amount >= 0)
);
```

DuckDB enforces `PRIMARY KEY`, `UNIQUE`, `NOT NULL`, `CHECK`, and `FOREIGN KEY` constraints. It also supports transactions with `BEGIN`, `COMMIT`, and `ROLLBACK`.

The practical difference is that MySQL/InnoDB provides a more complete operational environment for continuously changing application data.

## Capabilities commonly expected from MySQL/InnoDB OLTP systems

### 1. Many independent concurrent application writers

MySQL runs as a database server. Web services, background workers, and administrative tools can connect from separate processes or machines and perform transactions concurrently. InnoDB provides row-level locking, locking reads, lock waits, deadlock detection, and configurable transaction isolation.

```sql
START TRANSACTION;

SELECT available_quantity
FROM inventory
WHERE product_id = 42
FOR UPDATE;

UPDATE inventory
SET available_quantity = available_quantity - 1
WHERE product_id = 42;

COMMIT;
```

The `FOR UPDATE` read protects the selected inventory row while the transaction decides whether to modify it.

Standalone DuckDB is usually embedded in one process. It supports concurrent writers inside that process when their changes do not conflict, but it is not a drop-in replacement for a MySQL server receiving write transactions from many independent application processes.

### 2. Richer foreign-key actions

MySQL/InnoDB supports actions that automatically maintain related rows:

```sql
CREATE TABLE order_items (
    order_item_id BIGINT PRIMARY KEY AUTO_INCREMENT,
    order_id BIGINT NOT NULL,
    product_id BIGINT NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
        ON DELETE CASCADE,
    FOREIGN KEY (product_id)
        REFERENCES products(product_id)
        ON DELETE RESTRICT
);
```

DuckDB supports foreign-key validation, but cascading deletes such as `ON DELETE CASCADE` are not supported. Its foreign-key and index implementation also has documented limitations for some update and concurrent-transaction patterns.

### 3. Mature constraint changes on existing tables

In MySQL, constraints can be added or removed as a schema evolves:

```sql
ALTER TABLE orders
    ADD CONSTRAINT fk_orders_customer
    FOREIGN KEY (customer_id)
    REFERENCES customers(customer_id);
```

DuckDB supports many `ALTER TABLE` operations, but `ADD CONSTRAINT` and `DROP CONSTRAINT` are not currently supported. Constraints should generally be defined when the DuckDB table is created, or the table can be recreated with the required definition.

### 4. OLTP identity generation

MySQL/InnoDB provides established `AUTO_INCREMENT` behavior for generating keys during concurrent inserts:

```sql
CREATE TABLE orders (
    order_id BIGINT PRIMARY KEY AUTO_INCREMENT,
    customer_id BIGINT NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
) ENGINE = InnoDB;
```

DuckDB provides sequences and `nextval()` for generated numeric values, but its design target is analytical and batch processing rather than high-concurrency application identity generation.

### 5. Operational database services

A production MySQL deployment commonly includes capabilities around the database engine:

- network clients and connection management;
- users, roles, and server-side privileges;
- binary logging and replication;
- backups and point-in-time recovery workflows;
- monitoring of transactions, locks, waits, and deadlocks;
- high-availability deployment patterns.

DuckDB is primarily an embedded analytical engine. These server operations are not the main responsibility of a standalone DuckDB database file.

## MVCC: Multi-Version Concurrency Control

**MVCC stands for Multi-Version Concurrency Control.** It is a database-management technique that allows readers and writers to work concurrently while each transaction sees a consistent view of the data.

### A simple mental model

Instead of immediately destroying the previous value when a row changes, an MVCC database keeps enough version information to determine which value each transaction is allowed to see.

1. **A reader sees a snapshot.** A query observes data that is valid for its transaction snapshot. Changes committed after that snapshot are not allowed to appear halfway through the query.
2. **A writer creates a new version.** An update records a new row version associated with transaction metadata rather than making the old value disappear immediately.
3. **Commit controls visibility.** Other transactions see the new version only when the database's visibility rules allow it. A rollback discards the uncommitted change.
4. **Old versions are eventually reclaimed.** Once no active transaction needs an old version, the engine can remove or reuse it. The exact cleanup mechanism differs by database.

```text
Initial committed value:       inventory = 10
Transaction A starts reading:  snapshot sees 10
Transaction B updates value:   new version = 9
Transaction B commits:         new transactions can see 9
Transaction A continues:       its existing snapshot still sees 10
```

The important phrase is **reduced read/write blocking**, not **no blocking under every condition**. Readers can usually proceed without waiting for writers, but two transactions trying to modify the same row can still conflict. Locks, uniqueness checks, schema changes, and resource limits can also cause waiting or errors.

### MVCC in operational databases and DuckDB

| Concern | MySQL/InnoDB or PostgreSQL | Standalone DuckDB |
|---|---|---|
| Primary workload | Many short application transactions | Long scans, joins, aggregations, and batch changes |
| Concurrency environment | Many independent client processes connecting to a server | Commonly multiple connections or threads inside one process |
| Writer behavior | Designed for high numbers of concurrent row-level transactions | Non-conflicting writes can run concurrently inside one process; conflicting writes use optimistic conflict detection |
| Reader consistency | MVCC gives each transaction a consistent visible version | MVCC keeps an analytical query's view consistent while other in-process transactions commit changes |
| Old-version cleanup | Engine-specific: for example, InnoDB purges undo records and PostgreSQL uses vacuum | Old versions become reclaimable when active transactions no longer need them; disk-space reclamation can also depend on checkpointing |

### DuckDB example

Suppose a DuckDB analytical query takes ten seconds to scan and aggregate a large table. While it runs, another connection or thread **in the same DuckDB writer process** appends one million rows and commits. MVCC keeps the running query internally consistent: it continues using its established snapshot instead of seeing a mixture of the table before and after the append. A later transaction can see the newly committed rows.

> In standard standalone read-write mode, do not describe this as an unrelated process writing the same DuckDB database file. DuckDB's normal read-write concurrency is within a single process; multiple processes may open the file concurrently only in read-only mode.

## Both systems are transactional—but optimized for different shapes of work

```text
MySQL/InnoDB transaction:
    Read one customer → insert one order → update two inventory rows → commit

DuckDB analytical transaction:
    Scan Parquet → join millions of rows → aggregate → write a result dataset
```

Both can use SQL and transactions. The difference is the workload each system makes especially efficient.

## A practical architecture

An application does not always need to choose only one system:

```text
Customer activity
      ↓
MySQL/InnoDB
  authoritative operational records
      ↓  extract or replicate
Parquet files / analytical storage
      ↓
DuckDB
  local analysis, reporting, and transformations
```

Use **MySQL/InnoDB** when the central requirement is a shared operational database with many concurrent application transactions.

Use **DuckDB** when the central requirement is fast analytics over local files, Parquet datasets, Arrow objects, or in-process data.

Use both when MySQL owns the operational truth and DuckDB performs analytical work without burdening the transactional system.

## Documentation

- [DuckDB constraints](https://duckdb.org/docs/stable/sql/constraints)
- [DuckDB concurrency](https://duckdb.org/docs/stable/connect/concurrency)
- [DuckDB `CREATE TABLE` and foreign-key limitations](https://duckdb.org/docs/stable/sql/statements/create_table)
- [DuckDB `ALTER TABLE` limitations](https://duckdb.org/docs/stable/sql/statements/alter_table)
- [MySQL/InnoDB locking](https://dev.mysql.com/doc/refman/8.4/en/innodb-locking.html)
- [MySQL/InnoDB transaction isolation](https://dev.mysql.com/doc/refman/8.4/en/innodb-transaction-isolation-levels.html)
- [MySQL foreign-key constraints](https://dev.mysql.com/doc/refman/8.4/en/create-table-foreign-keys.html)


# DuckDB and Parquet

DuckDB is an in-process analytical SQL database. It can query Parquet files directly without first loading them into a database table or a pandas DataFrame.

In this notebook, DuckDB will:

1. Read the MovieLens `movies.parquet` and `ratings.parquet` files.
2. Find movies rated by at least 100 users.
3. Keep movies with an average rating of 4.0 or higher.
4. Join the aggregates to movie titles.
5. Write the result to `popular-movies.parquet`.
6. Read the saved Parquet file and validate it.

No pandas API is used.

## Installation

```powershell
python -m pip install duckdb
```


## 1. Import DuckDB and define file locations

The database connection uses `:memory:`, so it does not create a DuckDB database file. The source and result data remain in Parquet.


In [ ]:
from pathlib import Path

import duckdb

PARQUET_DIR = Path(r"C:\data\movielens\parquet")
MOVIES_PATH = PARQUET_DIR / "movies.parquet"
RATINGS_PATH = PARQUET_DIR / "ratings.parquet"
POPULAR_MOVIES_PATH = PARQUET_DIR / "popular-movies.parquet"

for source_path in (MOVIES_PATH, RATINGS_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(f"Required Parquet file not found: {source_path}")

connection = duckdb.connect(database=":memory:")

print(f"DuckDB version: {duckdb.__version__}")
print(f"Movies: {MOVIES_PATH}")
print(f"Ratings: {RATINGS_PATH}")
print(f"Output: {POPULAR_MOVIES_PATH}")


## 2. Small display helper

DuckDB returns tuples through `fetchall()`. This helper prints those native results with column headings and deliberately avoids pandas conversion methods.


In [ ]:
def show_query(sql, parameters=None, limit=None):
    result = connection.execute(sql, parameters or [])
    columns = [item[0] for item in result.description]
    rows = result.fetchmany(limit) if limit is not None else result.fetchall()
    print(" | ".join(columns))
    print("-" * 80)
    for row in rows:
        print(" | ".join(str(value) for value in row))
    return rows


def sql_string(path):
    # Return a safely quoted SQL string literal for a local path.
    return "'" + str(path).replace("'", "''") + "'"


## 3. Read Parquet files directly

`read_parquet()` is a DuckDB table function. DuckDB can push column selection and filters into the Parquet scan, reducing unnecessary I/O.

Here we inspect schemas and row counts without creating permanent database tables.


In [ ]:
movies_sql_path = sql_string(MOVIES_PATH)
ratings_sql_path = sql_string(RATINGS_PATH)

print("MOVIES SCHEMA")
show_query(f"DESCRIBE SELECT * FROM read_parquet({movies_sql_path})")

print("\nRATINGS SCHEMA")
show_query(f"DESCRIBE SELECT * FROM read_parquet({ratings_sql_path})")

movie_count = connection.execute(
    f"SELECT count(*) FROM read_parquet({movies_sql_path})"
).fetchone()[0]
rating_count = connection.execute(
    f"SELECT count(*) FROM read_parquet({ratings_sql_path})"
).fetchone()[0]

print(f"\nMovie rows: {movie_count:,}")
print(f"Rating rows: {rating_count:,}")


## 4. Preview movies and ratings

The query reads only the columns needed for the preview. `ORDER BY` makes the displayed result deterministic.


In [ ]:
print("MOVIES SAMPLE")
show_query(
    f"""
    SELECT movieId, title, genres
    FROM read_parquet({movies_sql_path})
    ORDER BY movieId
    LIMIT 5
    """
)

print("\nRATINGS SAMPLE")
show_query(
    f"""
    SELECT userId, movieId, rating, timestamp
    FROM read_parquet({ratings_sql_path})
    ORDER BY userId, movieId
    LIMIT 5
    """
)


## 5. Define the popular-movies query

The aggregation computes:

- `rating_count`: number of ratings received by a movie
- `distinct_user_count`: number of distinct users who rated it
- `average_rating`: mean score

The requirement is applied with `HAVING`: at least 100 distinct users and an average rating of at least 4.0. The result is joined to `movies.parquet` to add the title and genres.


In [ ]:
popular_movies_query = f"""
WITH rating_summary AS (
    SELECT
        movieId,
        count(*) AS rating_count,
        count(DISTINCT userId) AS distinct_user_count,
        avg(rating) AS average_rating
    FROM read_parquet({ratings_sql_path})
    GROUP BY movieId
    HAVING count(DISTINCT userId) >= 100
       AND avg(rating) >= 4.0
)
SELECT
    summary.movieId,
    movies.title,
    movies.genres,
    summary.rating_count,
    summary.distinct_user_count,
    round(summary.average_rating, 3) AS average_rating
FROM rating_summary AS summary
INNER JOIN read_parquet({movies_sql_path}) AS movies
    ON summary.movieId = movies.movieId
ORDER BY
    summary.average_rating DESC,
    summary.distinct_user_count DESC,
    summary.movieId
"""

popular_movies = connection.execute(popular_movies_query).fetchall()
popular_columns = [item[0] for item in connection.description]

print(" | ".join(popular_columns))
print("-" * 100)
for row in popular_movies:
    print(" | ".join(str(value) for value in row))

print(f"\nPopular movies found: {len(popular_movies)}")
assert popular_movies, "Expected at least one popular movie for the selected thresholds."


## 6. Write the query result to Parquet

DuckDB's `COPY` statement can write any query result directly to Parquet. ZSTD compression is selected for a compact analytical output.

The query is wrapped in parentheses because `COPY` expects a relation-producing statement.


In [ ]:
popular_movies_sql_path = sql_string(POPULAR_MOVIES_PATH)

connection.execute(
    f"""
    COPY ({popular_movies_query})
    TO {popular_movies_sql_path}
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

print(f"Created: {POPULAR_MOVIES_PATH}")
print(f"File size: {POPULAR_MOVIES_PATH.stat().st_size:,} bytes")


## 7. Read the saved result

The newly created file is queried directly. No import step or DataFrame is required.


In [ ]:
print("POPULAR MOVIES READ FROM PARQUET")
saved_rows = show_query(
    f"""
    SELECT
        movieId,
        title,
        genres,
        rating_count,
        distinct_user_count,
        average_rating
    FROM read_parquet({popular_movies_sql_path})
    ORDER BY average_rating DESC, distinct_user_count DESC, movieId
    """
)


## 8. Validate the output

The checks confirm that:

- every result has at least 100 distinct raters;
- every average rating is at least 4.0;
- the saved row count matches the query result;
- the output schema contains the expected columns.


In [ ]:
validation = connection.execute(
    f"""
    SELECT
        count(*) AS row_count,
        min(distinct_user_count) AS minimum_distinct_users,
        min(average_rating) AS minimum_average_rating
    FROM read_parquet({popular_movies_sql_path})
    """
).fetchone()

saved_schema = connection.execute(
    f"DESCRIBE SELECT * FROM read_parquet({popular_movies_sql_path})"
).fetchall()
saved_column_names = [row[0] for row in saved_schema]
expected_columns = [
    "movieId",
    "title",
    "genres",
    "rating_count",
    "distinct_user_count",
    "average_rating",
]

row_count, minimum_distinct_users, minimum_average_rating = validation

assert row_count == len(popular_movies)
assert minimum_distinct_users >= 100
assert minimum_average_rating >= 4.0
assert saved_column_names == expected_columns

print(f"Rows validated: {row_count}")
print(f"Minimum distinct users: {minimum_distinct_users}")
print(f"Minimum average rating: {minimum_average_rating}")
print("Output validation passed.")


## 9. Close the connection

Closing the in-memory connection releases its resources. The Parquet files remain available on disk.


In [ ]:
connection.close()
print("DuckDB connection closed.")


## Summary

- DuckDB queries Parquet files directly through `read_parquet()`.
- SQL aggregation, filtering, and joins do not require pandas.
- `COPY (query) TO ... (FORMAT PARQUET)` writes query results directly to Parquet.
- Column and filter pushdown allow DuckDB to avoid reading unnecessary Parquet data.
- The final dataset was written to:

`C:\data\movielens\parquet\popular-movies.parquet`
